In [2]:
# gerekli kütüphaneleri import edelim

import requests
import threading
import time
import asyncio
import aiohttp

In [3]:
urls = ["https://postman-echo.com/delay/3"] * 10 # 3 saniye delay'li 10 istek

In [4]:
# istekleri senkron atmayı deneyelim

def get_data_sync(urls):
    st = time.time() # işlem başladığında bilgisayarın zamanını almak
    json_data = []
    for url in urls:
        json_data.append(requests.get(url).json())
    et = time.time() # işlem bittiğinde zamanı tekrar almak
    elapsed_time = et - st # saniye cinsinden geçen süre
    print("Execution time : ", elapsed_time, "seconds.")
    return json_data

get_data_sync(urls) # sonuç : 51.8 saniye

Execution time :  51.88956665992737 seconds.


[{'delay': '3'},
 {'delay': '3'},
 {'delay': '3'},
 {'delay': '3'},
 {'delay': '3'},
 {'delay': '3'},
 {'delay': '3'},
 {'delay': '3'},
 {'delay': '3'},
 {'delay': '3'}]

In [5]:
# istekleri threadlere bölerek atmayı deneyelim
# bunu yapmak için Thread sınıfından miras alan bir sınıf yazmamız gerekir


class ThreadingDownloader(threading.Thread): # kalıtım
    json_data = []
    def __init__(self, url):
        super().__init__()
        self.url = url
    def run(self): # start fonksiyonu çağırıldığında buradaki run çalıştırılır
        response = requests.get(self.url)
        self.json_data.append(response.json())
        return self.json_data

def get_data_threading(urls):
    st = time.time()
    threads = []
    for url in urls: # thread'lerin bitmesini beklemeden sırayla çağırıyoruz
        t = ThreadingDownloader(url)
        t.start()
        threads.append(t)
    for t in threads:
        t.join() # bitmemiş thread'ler olabileceği için sırayla bitmesi beklenir, indirme işleminin tamamlandığından emin olunur
        print(t)
    et = time.time()
    elapsed_time = et - st
    print("Execution time : ", elapsed_time, "seconds.")

get_data_threading(urls) # sonuç: 5 saniye

<ThreadingDownloader(Thread-3, stopped 3212)>
<ThreadingDownloader(Thread-4, stopped 18660)>
<ThreadingDownloader(Thread-5, stopped 19508)>
<ThreadingDownloader(Thread-6, stopped 8076)>
<ThreadingDownloader(Thread-7, stopped 31836)>
<ThreadingDownloader(Thread-8, stopped 20932)>
<ThreadingDownloader(Thread-9, stopped 30872)>
<ThreadingDownloader(Thread-10, stopped 31224)>
<ThreadingDownloader(Thread-11, stopped 30744)>
<ThreadingDownloader(Thread-12, stopped 34496)>
Execution time :  5.098529577255249 seconds.


In [7]:
# istekleri asenkron atalım
# tüm istekler aynı anda atılır, ilk giren içeri alınır. toplam süre = en uzun süren işlemin süresi

async def get_data(url, session): # asenkron programlama yapabilmek için ve cevabın gelmesini beklememek için async - await kullanımı zorunludur
    async with session.get(url) as response:
        return await response.json()

async def get_data_async(urls):
    st = time.time()
    json_data = []
    async with aiohttp.ClientSession() as session:
        tasks = [asyncio.ensure_future(get_data(url, session)) for url in urls] # görev listesi oluşturulur
        json_data = await asyncio.gather(*tasks) # args sayesinde her task gather'la aynı anda çalıştırılır
    et = time.time()
    elapsed_time = et - st
    print("Execution time : ", elapsed_time, "seconds.")
    return json_data

# asyncio.run(get_data_async(urls)) normal .py dosyasında bu kodla çalıştırırken jupyter notebook zaten asenkron olduğu için başka bir şey deneyeceğiz
await get_data_async(urls) # 4.6 saniye ile en hızlı

Execution time :  4.622294664382935 seconds.


[{'delay': '3'},
 {'delay': '3'},
 {'delay': '3'},
 {'delay': '3'},
 {'delay': '3'},
 {'delay': '3'},
 {'delay': '3'},
 {'delay': '3'},
 {'delay': '3'},
 {'delay': '3'}]